# Диагностика классификаций

Тематический этап подготовки Европы 2019–2023. Выполняйте ячейки сверху вниз в окружении проекта. Каждый ноутбук соответствует одному шагу; диагностика и применение согласованного изменения разделены. Входы, выходы и порядок описаны в [карте подготовки](README.md). Сохраненные результаты перенесены из выполненного общего ноутбука; сам перенос данных не преобразует.

## Шаг 3 – Диагностика неоднозначных кодов самолетов

Читаем рабочую копию после согласованной нормализации пустых строк. Сравниваем `icaoaircrafttype` с преобладающим шаблоном `[A-Z][0-9][A-Z]` и с `typecode`. Это структурная проверка, а не справочник допустимых авиационных кодов. Верхний регистр используется только при сравнении, исходные значения не изменяются.

Показываем все непустые значения вне шаблона, пары с `typecode`, распределение по аэропортам и до трех исходных примеров на значение. Возможные соответствия внутри базы служат подсказкой для последующей проверки источника.

In [1]:
# Готовим диагностику классификаций самолетов в нормализованной копии
from pathlib import Path
from collections import Counter
import hashlib
import json
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'europe/data/source/osn23_landings_merged.parquet.gzip').exists())
WORKING = ROOT / 'europe/data/working/01_empty_strings_normalized.parquet'
REPORT = ROOT / 'europe/reports/preparation'
log = json.loads((REPORT / '01_empty_strings_normalized.json').read_text(encoding='utf8'))

def hash_file(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

# Проверяем, что вход соответствует журналу предыдущего шага
working_before = hash_file(WORKING)
assert working_before == log['output_sha256']
reader = pq.ParquetFile(WORKING)
pair_counts = Counter()
pair_events = Counter()
airport_counts = Counter()
consistent_pairs = Counter()
sample_counts = Counter()
samples = []
rows = flagged = matched_typecode = missing_class = 0
columns = ['flight_id', 'icao24', 'landing_time', 'airport', 'typecode',
           'icaoaircrafttype', 'GoA', 'n_attempts']
# Сканируем всю таблицу и собираем сочетания кодов, метки и примеры
for batch in reader.iter_batches(batch_size=100_000, columns=columns):
    frame = batch.to_pandas(ignore_metadata=True)
    classification = frame['icaoaircrafttype']
    # Шаблон выделяет значения для разбора, но сам по себе не доказывает ошибку
    valid_shape = classification.str.fullmatch(r'[A-Z][0-9][A-Z]', na=False)
    suspicious = classification.notna() & ~valid_shape
    missing_class += int(classification.isna().sum())
    matches = (classification.notna() & frame['typecode'].notna()
               & classification.str.upper().eq(frame['typecode'].str.upper())).fillna(False)
    matched_typecode += int(matches.sum())
    for record in frame.loc[valid_shape & frame['typecode'].notna(), ['typecode', 'icaoaircrafttype']].itertuples(index=False):
        consistent_pairs[(record.typecode, record.icaoaircrafttype)] += 1
    for position, record in frame.loc[suspicious].iterrows():
        code = record['icaoaircrafttype']
        typecode = None if pd.isna(record['typecode']) else record['typecode']
        pair = (code, typecode)
        pair_counts[pair] += 1
        pair_events[pair] += int(record['GoA'])
        airport_counts[(code, record['airport'])] += 1
        if sample_counts[code] < 3:
            sample = record.to_dict()
            sample['source_row_position'] = rows + int(position)
            sample['equals_typecode_ignoring_case'] = bool(matches.loc[position])
            samples.append(sample)
            sample_counts[code] += 1
    flagged += int(suspicious.sum())
    rows += len(frame)

# Проверяем охват и неизменность входа, затем строим диагностические таблицы
assert rows == log['rows_after']
assert hash_file(WORKING) == working_before
pairs = pd.DataFrame([{'icaoaircrafttype': code, 'typecode': typecode, 'rows': count,
                       'go_arounds': pair_events[(code, typecode)]}
                      for (code, typecode), count in pair_counts.items()]).sort_values('rows', ascending=False)
codes = pairs.groupby('icaoaircrafttype')[['rows', 'go_arounds']].sum().sort_values('rows', ascending=False)
codes['ga_percent'] = 100 * codes['go_arounds'] / codes['rows']
airport_distribution = pd.DataFrame([{'icaoaircrafttype': code, 'airport': airport, 'rows': count}
                                    for (code, airport), count in airport_counts.items()]).sort_values('rows', ascending=False)
comparison = pd.DataFrame([{'typecode': typecode, 'icaoaircrafttype': code, 'rows': count}
                           for (typecode, code), count in consistent_pairs.items()])
affected_types = pairs['typecode'].dropna().unique()
comparison = comparison.loc[comparison['typecode'].isin(affected_types)].sort_values(['typecode', 'rows'], ascending=[True, False])
# Сохраняем отчеты; значения в рабочей базе не исправляем
codes.to_csv(REPORT / '02_aircraft_code_values.csv')
pairs.to_csv(REPORT / '02_aircraft_code_pairs.csv', index=False)
airport_distribution.to_csv(REPORT / '02_aircraft_code_airports.csv', index=False)
comparison.to_csv(REPORT / '02_aircraft_code_internal_reference.csv', index=False)
print('Всего записей:', rows)
print('Классификация отсутствует:', missing_class)
print('Непустые значения вне преобладающего шаблона:', flagged)
print('Совпадают с typecode без учета регистра:', matched_typecode)
display(codes)
print('Рабочая копия не изменена. Исправления не применялись.')

Всего записей: 7984254
Классификация отсутствует: 1402236
Непустые значения вне преобладающего шаблона: 6407
Совпадают с typecode без учета регистра: 130


,rows,go_arounds,ga_percent
icaoaircrafttype,,,
A332,1140,1,0.087719
a320,1092,8,0.732601
A21N,705,7,0.992908
GLF6,692,5,0.722543
FA8X,561,6,1.069519
B734,481,0,0.000000
82R,457,2,0.437637
C68A,357,0,0.000000
GL5T,330,0,0.000000


Рабочая копия не изменена. Исправления не применялись.


In [2]:
# Сопоставляем подозрительные коды с typecode и распределением по аэропортам
print('Пары значений вне шаблона и typecode:')
display(pairs)
print('Распределение по аэропортам:')
display(airport_distribution)
# Внутренние сочетания служат сравнением, а не официальным справочником
print('Другие классификации тех же typecode внутри базы – только для сравнения:')
display(comparison)
# Показываем ограниченное число исходных строк для ручной проверки
print('Примеры записей, до трех на значение классификации:')
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(pd.DataFrame(samples))

Пары значений вне шаблона и typecode:


,icaoaircrafttype,typecode,rows,go_arounds
23,A332,NaN,1137,1
0,a320,NaN,1092,8
5,A21N,NaN,705,7
2,FA8X,Falcon 8X,561,6
6,GLF6,G650,495,5
12,B734,NaN,481,0
3,82R,737,457,2
1,C68A,680 Citation Latitude,357,0
4,GL5T,BD-700-1A11 Global 5000,330,0
8,GLF6,G650ER,197,0


Распределение по аэропортам:


,icaoaircrafttype,airport,rows
129,A332,LOWW,667
0,a320,LIRF,406
3,FA8X,LSGG,379
131,A332,EGSS,281
76,B734,LIMC,186
...,...,...,...
234,G-V,EHAM,1
236,G550,EDDM,1
237,A319,LFPG,1
239,F2TH,LIMC,1


Другие классификации тех же typecode внутри базы – только для сравнения:


,typecode,icaoaircrafttype,rows
171,G550,L2J,12
314,G650,L2J,1
224,G650ER,L2J,3


Примеры записей, до трех на значение классификации:


,flight_id,icao24,landing_time,airport,typecode,icaoaircrafttype,GoA,n_attempts,source_row_position,equals_typecode_ignoring_case
0,VLG442_25252,345643,2019-01-01 07:49:29+00:00,LIRF,NaN,a320,False,1,661,False
1,VLG65UY_25306,345643,2019-01-01 15:49:42+00:00,LIRF,NaN,a320,False,1,2990,False
2,VLG6137_25261,345643,2019-01-01 19:44:15+00:00,LIRF,NaN,a320,False,1,4392,False
3,ASJ794_7181,398275,2019-01-02 11:24:21+00:00,LSGG,680 Citation Latitude,C68A,False,1,7476,False
4,ASJ794_7182,398275,2019-01-02 16:31:36+00:00,LSGG,680 Citation Latitude,C68A,False,1,9266,False
5,LUC17_51290,4b1a17,2019-01-04 20:37:15+00:00,LSGG,Falcon 8X,FA8X,False,1,23249,False
6,BMS4H_59456,4a09ae,2019-01-05 17:56:33+00:00,ESSA,737,82R,False,1,28147,False
7,ULC91_72659,4d2188,2019-01-05 21:19:49+00:00,LSGG,BD-700-1A11 Global 5000,GL5T,False,1,29173,False
8,BMS6TT_146606,4a09ae,2019-01-06 16:23:53+00:00,LEMD,737,82R,False,1,33371,False
9,ASJ312_7668,39c8a9,2019-01-07 11:03:35+00:00,LSGG,680 Citation Latitude,C68A,False,1,37967,False


## Согласование следующего правила

Исправления кодов не применены. Выход за преобладающий шаблон и совпадение двух полей не доказывают конкретное правильное значение. Следующее правило согласуем после просмотра групп и проверки происхождения полей. Нормализация регистра, перенос кодов между полями, замена на null и удаление строк здесь не выполняются.